# Using the singlekwargdispatch Decorator in baseobjects

## Introduction

The `singlekwargdispatch` decorator extends Python's built-in `singledispatch` functionality to allow keyword arguments to be used for dispatching. While the standard `singledispatch` requires at least one positional argument for dispatching, `singlekwargdispatch` retains this functionality but also allows the first keyword argument to be used for dispatching if no positional arguments are provided. Additionally, you can specify a particular keyword argument name to be used for dispatching.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `singlekwargdispatch` decorator
- Learning how to use basic positional argument dispatching
- Using keyword argument dispatching
- Creating methods with `singlekwargdispatch`
- Exploring advanced features and use cases

**Prerequisites:**
- Basic understanding of Python decorators
- Familiarity with Python's `singledispatch` functionality
- Understanding of function annotations and type hints

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import singlekwargdispatch

## Core Functionality

The `singlekwargdispatch` decorator extends Python's built-in `singledispatch` functionality to allow keyword arguments to be used for dispatching. Let's explore the core functionality and understand how it works.

### Basic Concept

The standard `singledispatch` from the `functools` module allows you to create a function that dispatches to different implementations based on the type of its first argument. However, it has a limitation: it requires at least one positional argument for dispatching.

`singlekwargdispatch` addresses this limitation by:
1. Retaining the ability to dispatch based on the first positional argument
2. Allowing dispatching based on the first keyword argument if no positional arguments are provided
3. Providing the option to specify a particular keyword argument to use for dispatching

Let's create some simple functions to demonstrate:

In [2]:
# Basic usage with positional argument dispatching
@singlekwargdispatch
def process_value(value) -> str:
    """Process a value of any type.

    This is the default implementation that handles any type.

    Args:
        value: The value to process.

    Returns:
        A string describing the processed value.
    """
    return f"Default: {value}"


@process_value.register
def _(value: int) -> str:
    """Process an integer value.

    Args:
        value: The integer to process.

    Returns:
        A string describing the processed integer.
    """
    return f"Integer: {value} (squared = {value ** 2})"


@process_value.register
def _(value: str) -> str:
    """Process a string value.

    Args:
        value: The string to process.

    Returns:
        A string describing the processed string.
    """
    return f"String: '{value}' (length = {len(value)})"


@process_value.register
def _(value: list) -> str:
    """Process a list value.

    Args:
        value: The list to process.

    Returns:
        A string describing the processed list.
    """
    return f"List: {value} (length = {len(value)})"


# Test the function with different types
print("Testing positional argument dispatching:")
print(process_value(42))
print(process_value("Hello, world!"))
print(process_value([1, 2, 3, 4, 5]))
print(process_value(3.14159))  # Uses default implementation

Testing positional argument dispatching:
Integer: 42 (squared = 1764)
String: 'Hello, world!' (length = 13)
List: [1, 2, 3, 4, 5] (length = 5)
Default: 3.14159


### Keyword Argument Dispatching

Now let's explore how `singlekwargdispatch` allows dispatching based on keyword arguments. We can create a function that uses the first keyword argument for dispatching if no positional arguments are provided:

In [3]:
# Using the first keyword argument for dispatching
@singlekwargdispatch
def format_data(data=None, **kwargs) -> str:
    """Format data of any type.

    This is the default implementation that handles any type.

    Args:
        data: The data to format.
        **kwargs: Additional keyword arguments.

    Returns:
        A string with the formatted data.
    """
    return f"Default format: {data}"


@format_data.register
def _(data: int | None = None, **kwargs) -> str:
    """Format an integer.

    Args:
        data: The integer to format.
        **kwargs: Additional keyword arguments.

    Returns:
        A string with the formatted integer.
    """
    return f"Integer format: {data} (hex = {hex(data)})"


@format_data.register
def _(data: str | None = None, **kwargs) -> str:
    """Format a string.

    Args:
        data: The string to format.
        **kwargs: Additional keyword arguments.

    Returns:
        A string with the formatted string.
    """
    return f"String format: '{data}' (uppercase = '{data.upper()}')"


# Test the function with keyword arguments
print("\nTesting keyword argument dispatching:")
print(format_data(data=42))
print(format_data(data="hello"))
print(format_data(data=3.14159))  # Uses default implementation

# Test with positional arguments (still works like singledispatch)
print("\nTesting with positional arguments:")
print(format_data(42))
print(format_data("hello"))


Testing keyword argument dispatching:
Integer format: 42 (hex = 0x2a)
String format: 'hello' (uppercase = 'HELLO')
Default format: 3.14159

Testing with positional arguments:
Integer format: 42 (hex = 0x2a)
String format: 'hello' (uppercase = 'HELLO')


### Specifying a Keyword Argument for Dispatching

`singlekwargdispatch` also allows you to specify a particular keyword argument to use for dispatching. This is useful when you want to dispatch based on a specific parameter that might not be the first one:

In [4]:
# Specifying a keyword argument for dispatching
@singlekwargdispatch(kwarg="format_as")
def convert_value(value, format_as=None, precision=2) -> str:
    """Convert a value to a specific format.

    This is the default implementation that handles any format type.

    Args:
        value: The value to convert.
        format_as: The format type to use.
        precision: The precision for numeric formatting.

    Returns:
        The converted value as a string.
    """
    return f"Default format: {value} (precision = {precision})"


@convert_value.register
def _(value, format_as: str | None = None, precision=2):
    """Convert a value using a string format type.

    Args:
        value: The value to convert.
        format_as: The string format type.
        precision: The precision for numeric formatting.

    Returns:
        The converted value as a string.
    """
    if format_as.lower() == "uppercase":
        return str(value).upper()
    elif format_as.lower() == "lowercase":
        return str(value).lower()
    else:
        return f"Unknown string format: {format_as}"


@convert_value.register
def _(value, format_as: int | None = None, precision=2):
    """Convert a value using an integer format type.

    Args:
        value: The value to convert.
        format_as: The integer format type (1=decimal, 2=hex, 3=binary).
        precision: The precision for numeric formatting.

    Returns:
        The converted value as a string.
    """
    if isinstance(value, (int, float)):
        if format_as == 1:
            return f"{value:.{precision}f}"
        elif format_as == 2:
            return hex(int(value))
        elif format_as == 3:
            return bin(int(value))
        else:
            return f"Unknown integer format: {format_as}"
    else:
        return f"Cannot apply numeric formatting to {type(value).__name__}"


# Test the function with the specified keyword argument
print("\nTesting with specified keyword argument:")
print(convert_value(42, format_as="uppercase"))
print(convert_value("Mixed CASE", format_as="lowercase"))
print(convert_value(42, format_as=2))  # Hex format
print(convert_value(15, format_as=3))  # Binary format
print(convert_value(3.14159, format_as=1, precision=4))  # Decimal with precision


Testing with specified keyword argument:
42
mixed case
0x2a
0b1111
3.1416


## Module Interaction

The `singlekwargdispatch` decorator is part of the `baseobjects.functions` package and interacts with other components of the baseobjects package. Let's explore these interactions and see how `singlekwargdispatch` can be used with classes and methods.

### Relationship with BaseDecorator

`singlekwargdispatch` inherits from `BaseDecorator`, which is the base class for decorators in the baseobjects package. This provides a consistent interface and behavior across the package.

In [5]:
from baseobjects.functions import BaseDecorator
from typing import NoReturn

# Check inheritance
print(f"singlekwargdispatch is a subclass of BaseDecorator: {issubclass(singlekwargdispatch, BaseDecorator)}")

singlekwargdispatch is a subclass of BaseDecorator: True


### Using singlekwargdispatch with Methods

One of the most powerful features of `singlekwargdispatch` is its ability to work with methods in classes. Let's create a class that uses `singlekwargdispatch` to process different types of data:

In [6]:
class DataProcessor:
    """A class that processes data using singlekwargdispatch."""

    @singlekwargdispatch
    def process(self, data) -> str:
        """Process data of any type.

        This is the default implementation that handles any type.

        Args:
            data: The data to process.

        Returns:
            A string describing the processed data.
        """
        return f"Default processing: {data}"

    @process.register
    def _(self, data: int) -> str:
        """Process an integer.

        Args:
            data: The integer to process.

        Returns:
            A string describing the processed integer.
        """
        return f"Integer processing: {data} (doubled = {data * 2})"

    @process.register
    def _(self, data: str) -> str:
        """Process a string.

        Args:
            data: The string to process.

        Returns:
            A string describing the processed string.
        """
        return f"String processing: '{data}' (reversed = '{data[::-1]}')"

    @process.register
    def _(self, data: list) -> str:
        """Process a list.

        Args:
            data: The list to process.

        Returns:
            A string describing the processed list.
        """
        return f"List processing: {data} (sorted = {sorted(data)})"

    @singlekwargdispatch(kwarg="format_type")
    def format(self, data, format_type=None, **kwargs) -> str:
        """Format data with a specific format type.

        This is the default implementation that handles any format type.

        Args:
            data: The data to format.
            format_type: The type of formatting to apply.
            **kwargs: Additional keyword arguments.

        Returns:
            The formatted data as a string.
        """
        return f"Default formatting: {data}"

    @format.register
    def _(self, data, format_type: str | None = None, **kwargs):
        """Format data with a string format type.

        Args:
            data: The data to format.
            format_type: The string format type.
            **kwargs: Additional keyword arguments.

        Returns:
            The formatted data as a string.
        """
        if format_type.lower() == "json":
            import json
            try:
                return f"JSON formatting: {json.dumps(data, **kwargs)}"
            except (TypeError, ValueError):
                return f"Cannot convert {type(data).__name__} to JSON"
        elif format_type.lower() == "xml":
            # Simple XML formatting for demonstration
            if isinstance(data, dict):
                xml = ["<root>"]
                for key, value in data.items():
                    xml.append(f"  <{key}>{value}</{key}>")
                xml.append("</root>")
                return "XML formatting:\n" + "\n".join(xml)
            else:
                return f"Cannot convert {type(data).__name__} to XML"
        else:
            return f"Unknown string format type: {format_type}"

    @format.register
    def _(self, data, format_type: int | None = None, **kwargs) -> str:
        """Format data with an integer format type.

        Args:
            data: The data to format.
            format_type: The integer format type.
            **kwargs: Additional keyword arguments.

        Returns:
            The formatted data as a string.
        """
        padding = kwargs.get("padding", 0)
        if format_type == 1:  # Left-aligned
            return f"{data!s:<{padding}}"
        elif format_type == 2:  # Right-aligned
            return f"{data!s:>{padding}}"
        elif format_type == 3:  # Center-aligned
            return f"{data!s:^{padding}}"
        else:
            return f"Unknown integer format type: {format_type}"


# Create an instance of DataProcessor
processor = DataProcessor()

# Test the process method with different types
print("Testing process method with different types:")
print(processor.process(42))
print(processor.process("Hello, world!"))
print(processor.process([3, 1, 4, 1, 5, 9, 2, 6, 5]))
print(processor.process(3.14159))  # Uses default implementation

# Test the format method with different format types
print("\nTesting format method with different format types:")
print(processor.format({"name": "John", "age": 30}, format_type="json", indent=2))
print(processor.format({"name": "John", "age": 30}, format_type="xml"))
print(processor.format("Hello", format_type=1, padding=10))  # Left-aligned
print(processor.format("Hello", format_type=2, padding=10))  # Right-aligned
print(processor.format("Hello", format_type=3, padding=10))  # Center-aligned

Testing process method with different types:
Integer processing: 42 (doubled = 84)
String processing: 'Hello, world!' (reversed = '!dlrow ,olleH')
List processing: [3, 1, 4, 1, 5, 9, 2, 6, 5] (sorted = [1, 1, 2, 3, 4, 5, 5, 6, 9])
Default processing: 3.14159

Testing format method with different format types:
JSON formatting: {
  "name": "John",
  "age": 30
}
XML formatting:
<root>
  <name>John</name>
  <age>30</age>
</root>
Hello     
     Hello
  Hello   


### Using singlekwargdispatch with Inheritance

`singlekwargdispatch` also works well with inheritance. When a subclass inherits a method decorated with `singlekwargdispatch`, it can override specific implementations or add new ones:

In [7]:
class Shape:
    """Base class for shapes."""


class Circle(Shape):
    """A circle shape."""

    def __init__(self, radius) -> None:
        self.radius = radius

    def __str__(self) -> str:
        return f"Circle(radius={self.radius})"


class Rectangle(Shape):
    """A rectangle shape."""

    def __init__(self, width, height) -> None:
        self.width = width
        self.height = height

    def __str__(self) -> str:
        return f"Rectangle(width={self.width}, height={self.height})"


class Triangle(Shape):
    """A triangle shape."""

    def __init__(self, base, height) -> None:
        self.base = base
        self.height = height

    def __str__(self) -> str:
        return f"Triangle(base={self.base}, height={self.height})"


class ShapeProcessor:
    """A class that processes shapes using singlekwargdispatch."""

    @singlekwargdispatch
    def calculate_area(self, shape) -> NoReturn:
        """Calculate the area of a shape.

        This is the default implementation that raises NotImplementedError.

        Args:
            shape: The shape to calculate the area of.

        Raises:
            NotImplementedError: If the shape type is not supported.
        """
        msg = f"Area calculation not implemented for {type(shape).__name__}"
        raise NotImplementedError(msg)

    @calculate_area.register
    def _(self, shape: Circle):
        """Calculate the area of a circle.

        Args:
            shape: The circle to calculate the area of.

        Returns:
            The area of the circle.
        """
        import math
        return math.pi * shape.radius ** 2

    @calculate_area.register
    def _(self, shape: Rectangle):
        """Calculate the area of a rectangle.

        Args:
            shape: The rectangle to calculate the area of.

        Returns:
            The area of the rectangle.
        """
        return shape.width * shape.height

    @calculate_area.register
    def _(self, shape: Triangle):
        """Calculate the area of a triangle.

        Args:
            shape: The triangle to calculate the area of.

        Returns:
            The area of the triangle.
        """
        return 0.5 * shape.base * shape.height


class EnhancedShapeProcessor(ShapeProcessor):
    """A subclass that extends ShapeProcessor with additional functionality."""

    # Create a copy of the dispatcher to keep the implementations separate
    calculate_area = ShapeProcessor.calculate_area.deepcopy()

    # Override the implementation for Circle
    @calculate_area.register
    def _(self, shape: Circle) -> str:
        """Calculate the area of a circle with enhanced precision.

        Args:
            shape: The circle to calculate the area of.

        Returns:
            The area of the circle with enhanced precision.
        """
        import math
        area = math.pi * shape.radius ** 2
        return f"Enhanced circle area: {area:.6f}"

    # Add a new implementation for a new shape type
    class Ellipse(Shape):
        """An ellipse shape."""

        def __init__(self, a, b) -> None:
            self.a = a  # Semi-major axis
            self.b = b  # Semi-minor axis

        def __str__(self) -> str:
            return f"Ellipse(a={self.a}, b={self.b})"

    @calculate_area.register
    def _(self, shape: Ellipse):
        """Calculate the area of an ellipse.

        Args:
            shape: The ellipse to calculate the area of.

        Returns:
            The area of the ellipse.
        """
        import math
        return math.pi * shape.a * shape.b


# Create shapes
circle = Circle(5)
rectangle = Rectangle(4, 6)
triangle = Triangle(3, 8)

# Create processors
basic_processor = ShapeProcessor()
enhanced_processor = EnhancedShapeProcessor()
ellipse = EnhancedShapeProcessor.Ellipse(3, 2)

# Test with basic processor
print("\nTesting with basic processor:")
print(f"Area of {circle} = {basic_processor.calculate_area(circle)}")
print(f"Area of {rectangle} = {basic_processor.calculate_area(rectangle)}")
print(f"Area of {triangle} = {basic_processor.calculate_area(triangle)}")

# Test with enhanced processor
print("\nTesting with enhanced processor:")
print(f"Area of {circle} = {enhanced_processor.calculate_area(circle)}")  # Uses overridden implementation
print(f"Area of {rectangle} = {enhanced_processor.calculate_area(rectangle)}")  # Uses inherited implementation
print(f"Area of {triangle} = {enhanced_processor.calculate_area(triangle)}")  # Uses inherited implementation
print(f"Area of {ellipse} = {enhanced_processor.calculate_area(ellipse)}")  # Uses new implementation

# Try with unsupported shape
try:
    basic_processor.calculate_area(Shape())
except NotImplementedError as e:
    print(f"\nError with unsupported shape: {e}")


Testing with basic processor:
Area of Circle(radius=5) = 78.53981633974483
Area of Rectangle(width=4, height=6) = 24
Area of Triangle(base=3, height=8) = 12.0

Testing with enhanced processor:
Area of Circle(radius=5) = Enhanced circle area: 78.539816
Area of Rectangle(width=4, height=6) = 24
Area of Triangle(base=3, height=8) = 12.0
Area of Ellipse(a=3, b=2) = 18.84955592153876

Error with unsupported shape: Area calculation not implemented for Shape


## Advanced Features

Now let's explore some advanced features and use cases of `singlekwargdispatch`.

### Working with Union Types

`singlekwargdispatch` supports dispatching based on Union types, allowing you to register a single implementation for multiple types:

In [8]:
from typing import Union


@singlekwargdispatch
def process_numeric(value) -> str:
    """Process a numeric value.

    This is the default implementation that handles any type.

    Args:
        value: The value to process.

    Returns:
        A string describing the processed value.
    """
    return f"Default: {value}"


@process_numeric.register
def _(value: int | float) -> str:
    """Process an integer or float value.

    Args:
        value: The integer or float to process.

    Returns:
        A string describing the processed value.
    """
    return f"Numeric: {value} (doubled = {value * 2})"


@process_numeric.register
def _(value: list | tuple) -> str:
    """Process a list or tuple of values.

    Args:
        value: The list or tuple to process.

    Returns:
        A string describing the processed value.
    """
    return f"Sequence: {value} (sum = {sum(value)})"


# Test with different types
print("Testing with Union types:")
print(process_numeric(42))  # int
print(process_numeric(3.14159))  # float
print(process_numeric([1, 2, 3, 4, 5]))  # list
print(process_numeric((1, 2, 3, 4, 5)))  # tuple
print(process_numeric("hello"))  # Uses default implementation

Testing with Union types:
Numeric: 42 (doubled = 84)
Numeric: 3.14159 (doubled = 6.28318)
Sequence: [1, 2, 3, 4, 5] (sum = 15)
Sequence: (1, 2, 3, 4, 5) (sum = 15)
Default: hello


### Working with None Type

`singlekwargdispatch` also supports dispatching based on the None type, which can be useful for handling optional parameters:

In [9]:
from types import NoneType


@singlekwargdispatch
def process_optional(value) -> str:
    """Process a value that might be None.

    This is the default implementation that handles any type.

    Args:
        value: The value to process.

    Returns:
        A string describing the processed value.
    """
    return f"Default: {value}"


@process_optional.register
def _(value: NoneType) -> str:
    """Process a None value.

    Args:
        value: The None value to process.

    Returns:
        A string describing the processed value.
    """
    return "None value: No data provided"


@process_optional.register
def _(value: int) -> str:
    """Process an integer value.

    Args:
        value: The integer to process.

    Returns:
        A string describing the processed value.
    """
    return f"Integer: {value}"


# Test with different types including None
print("\nTesting with None type:")
print(process_optional(None))
print(process_optional(42))
print(process_optional("hello"))  # Uses default implementation


Testing with None type:
None value: No data provided
Integer: 42
Default: hello


### Working with Coroutines

`singlekwargdispatch` works seamlessly with coroutines, allowing you to create async functions with type-based dispatching:

In [10]:
import asyncio

# Import nest_asyncio to allow coroutines in Jupyter Notebooks
import nest_asyncio
nest_asyncio.apply()


@singlekwargdispatch
async def process_async(value) -> str:
    """Process a value asynchronously.

    This is the default implementation that handles any type.

    Args:
        value: The value to process.

    Returns:
        A string describing the processed value.
    """
    await asyncio.sleep(0.1)  # Simulate some async work
    return f"Default async: {value}"


@process_async.register
async def _(value: int) -> str:
    """Process an integer value asynchronously.

    Args:
        value: The integer to process.

    Returns:
        A string describing the processed value.
    """
    await asyncio.sleep(0.1)  # Simulate some async work
    return f"Integer async: {value} (squared = {value ** 2})"


@process_async.register
async def _(value: str) -> str:
    """Process a string value asynchronously.

    Args:
        value: The string to process.

    Returns:
        A string describing the processed value.
    """
    await asyncio.sleep(0.1)  # Simulate some async work
    return f"String async: '{value}' (uppercase = '{value.upper()}')"


# Define a function to run the async tests
async def run_async_tests() -> None:
    print("\nTesting with coroutines:")
    print(await process_async(42))
    print(await process_async("hello"))
    print(await process_async(3.14159))  # Uses default implementation


# Run the async tests
asyncio.run(run_async_tests())


Testing with coroutines:
Integer async: 42 (squared = 1764)
String async: 'hello' (uppercase = 'HELLO')
Default async: 3.14159


### Handling Edge Cases

Let's explore some edge cases and how `singlekwargdispatch` handles them:

In [11]:
# Edge case 1: No arguments provided
@singlekwargdispatch
def no_args_function() -> str:
    """A function with no arguments.

    This will raise a TypeError when called because there are no arguments to dispatch on.
    """
    return "This won't be reached"


# Edge case 2: Multiple dispatch with the same type
@singlekwargdispatch
def multiple_dispatch(value) -> str:
    """A function with multiple implementations for the same type.

    The last registered implementation will be used.
    """
    return f"Default: {value}"


@multiple_dispatch.register
def _(value: int) -> str:
    """First implementation for int."""
    return f"Integer (first): {value}"


@multiple_dispatch.register
def _(value: int) -> str:
    """Second implementation for int."""
    return f"Integer (second): {value}"


# Edge case 3: Registering a non-class type
@singlekwargdispatch
def non_class_dispatch(value) -> str:
    """A function that tries to register a non-class type.

    This will raise a TypeError when registering.
    """
    return f"Default: {value}"


# Test edge cases
print("\nTesting edge cases:")

# Edge case 1: No arguments provided
try:
    no_args_function()
except TypeError as e:
    print(f"No args function error: {e}")

# Edge case 2: Multiple dispatch with the same type
print(multiple_dispatch(42))  # Should use the second implementation

# Edge case 3: Registering a non-class type
try:
    # Create a non-class object to use as an annotation
    not_a_class = "not a class"
    non_class_dispatch.register(not_a_class, lambda x: "Invalid")
except TypeError as e:
    print(f"Non-class dispatch error: {e}")


Testing edge cases:
No args function error: No args or kwargs given to dispatch.
Integer (second): 42
Non-class dispatch error: Invalid first argument to `registry()`. 'not a class' is not a class or union type.


## Examples

Let's explore some practical examples of how `singlekwargdispatch` can be used in real-world scenarios.

### Example 1: Data Conversion System

In this example, we'll create a system for converting data between different formats:

In [12]:
from typing import Any, NoReturn


@singlekwargdispatch
def convert_value(value: Any) -> str:
    """Convert a value to a string representation.

    This is the default implementation that converts any value to a string.

    Args:
        value: The value to convert.

    Returns:
        The string representation of the value.
    """
    return f"Default: {value!s}"


@convert_value.register
def _(value: int) -> str:
    """Convert an integer to a string representation.

    Args:
        value: The integer to convert.

    Returns:
        The string representation of the integer.
    """
    return f"Integer: {value} (hex: {hex(value)})"


@convert_value.register
def _(value: float) -> str:
    """Convert a float to a string representation.

    Args:
        value: The float to convert.

    Returns:
        The string representation of the float.
    """
    return f"Float: {value:.4f} (scientific: {value:.2e})"


@convert_value.register
def _(value: str) -> str:
    """Convert a string to a formatted representation.

    Args:
        value: The string to convert.

    Returns:
        The formatted representation of the string.
    """
    return f"String: '{value}' (length: {len(value)})"


# Register the same implementation for both tuple and list
@convert_value.register(tuple)
@convert_value.register(list)
def _(value) -> str:
    """Convert a sequence to a string representation.

    Args:
        value: The sequence to convert.

    Returns:
        The string representation of the sequence.
    """
    return f"Sequence: {value} (length: {len(value)}, sum: {sum(value) if all(isinstance(x, (int, float)) for x in value) else 'N/A'})"


# Test the conversion system
print("Data Conversion System:")
print(convert_value(42))
print(convert_value(3.14159))
print(convert_value("Hello, world!"))
print(convert_value([1, 2, 3, 4, 5]))
print(convert_value((1, 2, 3, 4, 5)))
print(convert_value({"name": "John", "age": 30}))  # Uses default implementation

Data Conversion System:
Integer: 42 (hex: 0x2a)
Float: 3.1416 (scientific: 3.14e+00)
String: 'Hello, world!' (length: 13)
Sequence: [1, 2, 3, 4, 5] (length: 5, sum: 15)
Sequence: (1, 2, 3, 4, 5) (length: 5, sum: 15)
Default: {'name': 'John', 'age': 30}


### Example 2: Command Processor

In this example, we'll create a command processor that handles different types of commands:

In [13]:
class Command:
    """Base class for commands."""


class HelpCommand(Command):
    """Command to display help information."""

    def __init__(self, topic=None) -> None:
        self.topic = topic

    def __str__(self) -> str:
        return f"HelpCommand(topic={self.topic})"


class QuitCommand(Command):
    """Command to quit the application."""

    def __str__(self) -> str:
        return "QuitCommand()"


class PrintCommand(Command):
    """Command to print a message."""

    def __init__(self, message) -> None:
        self.message = message

    def __str__(self) -> str:
        return f"PrintCommand(message='{self.message}')"


class CommandProcessor:
    """A class that processes commands using singlekwargdispatch."""

    @singlekwargdispatch
    def execute(self, command) -> NoReturn:
        """Execute a command.

        This is the default implementation that raises an error for unknown commands.

        Args:
            command: The command to execute.

        Raises:
            ValueError: If the command type is not supported.
        """
        msg = f"Unknown command type: {type(command).__name__}"
        raise ValueError(msg)

    @execute.register
    def _(self, command: HelpCommand) -> str:
        """Execute a help command.

        Args:
            command: The help command to execute.

        Returns:
            The result of executing the help command.
        """
        if command.topic is None:
            return "Available commands: help, quit, print"
        elif command.topic == "help":
            return "Use 'help [topic]' to get help on a specific topic."
        elif command.topic == "quit":
            return "Use 'quit' to exit the application."
        elif command.topic == "print":
            return "Use 'print [message]' to print a message."
        else:
            return f"No help available for topic: {command.topic}"

    @execute.register
    def _(self, command: QuitCommand) -> str:
        """Execute a quit command.

        Args:
            command: The quit command to execute.

        Returns:
            The result of executing the quit command.
        """
        return "Quitting application..."

    @execute.register
    def _(self, command: PrintCommand) -> str:
        """Execute a print command.

        Args:
            command: The print command to execute.

        Returns:
            The result of executing the print command.
        """
        return f"Printing: {command.message}"


# Create a command processor
processor = CommandProcessor()

# Test with different commands
print("\nCommand Processor:")
print(processor.execute(HelpCommand()))
print(processor.execute(HelpCommand("print")))
print(processor.execute(QuitCommand()))
print(processor.execute(PrintCommand("Hello, world!")))

# Test with an unknown command
try:
    processor.execute(Command())
except ValueError as e:
    print(f"Error: {e}")


Command Processor:
Available commands: help, quit, print
Use 'print [message]' to print a message.
Quitting application...
Printing: Hello, world!
Error: Unknown command type: Command


### Example 3: Data Validation System

In this example, we'll create a data validation system that validates different types of data:

In [14]:
@singlekwargdispatch(kwarg="data_type")
def validate_data(data, data_type=None, **options):
    """Validate data based on its type.

    This is the default implementation that returns a generic validation result.

    Args:
        data: The data to validate.
        data_type: The type of data to validate.
        **options: Additional validation options.

    Returns:
        A tuple of (is_valid, message).
    """
    return (True, f"No specific validation for {data_type}")


@validate_data.register
def _(data, data_type: str = "email", **options):
    """Validate an email address.

    Args:
        data: The email address to validate.
        data_type: The type of data (should be "email").
        **options: Additional validation options.

    Returns:
        A tuple of (is_valid, message).
    """
    import re
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    if re.match(pattern, data):
        return (True, "Valid email address")
    else:
        return (False, "Invalid email address")


@validate_data.register
def _(data, data_type: str = "phone", **options):
    """Validate a phone number.

    Args:
        data: The phone number to validate.
        data_type: The type of data (should be "phone").
        **options: Additional validation options.

    Returns:
        A tuple of (is_valid, message).
    """
    import re
    pattern = r"^\+?[0-9]{10,15}$"
    if re.match(pattern, data):
        return (True, "Valid phone number")
    else:
        return (False, "Invalid phone number")


@validate_data.register
def _(data, data_type: int | None = None, **options):
    """Validate a numeric range.

    Args:
        data: The number to validate.
        data_type: The validation type (1=positive, 2=negative, 3=range).
        **options: Additional validation options.

    Returns:
        A tuple of (is_valid, message).
    """
    if data_type == 1:  # Positive number
        if isinstance(data, (int, float)) and data > 0:
            return (True, "Valid positive number")
        else:
            return (False, "Not a positive number")
    elif data_type == 2:  # Negative number
        if isinstance(data, (int, float)) and data < 0:
            return (True, "Valid negative number")
        else:
            return (False, "Not a negative number")
    elif data_type == 3:  # Range
        min_val = options.get("min", float("-inf"))
        max_val = options.get("max", float("inf"))
        if isinstance(data, (int, float)) and min_val <= data <= max_val:
            return (True, f"Number is within range [{min_val}, {max_val}]")
        else:
            return (False, f"Number is outside range [{min_val}, {max_val}]")
    else:
        return (True, "No specific numeric validation")


# Test the validation system
print("\nData Validation System:")
print(validate_data("user@example.com", data_type="email"))
print(validate_data("invalid-email", data_type="email"))
print(validate_data("+1234567890", data_type="phone"))
print(validate_data("123", data_type="phone"))
print(validate_data(42, data_type=1))  # Positive number
print(validate_data(-42, data_type=2))  # Negative number
print(validate_data(42, data_type=3, min=0, max=100))  # Range check
print(validate_data(142, data_type=3, min=0, max=100))  # Range check (outside)
print(validate_data("some data", data_type="unknown"))  # Unknown type


Data Validation System:
(False, 'Invalid phone number')
(False, 'Invalid phone number')
(True, 'Valid phone number')
(False, 'Invalid phone number')
(True, 'Valid positive number')
(True, 'Valid negative number')
(True, 'Number is within range [0, 100]')
(False, 'Number is outside range [0, 100]')
(False, 'Invalid phone number')


## API Highlights

The `singlekwargdispatch` decorator provides a rich API for creating functions that dispatch to different implementations based on the type of their arguments. Here's a summary of the key components:

### Class Definition

```python
class singlekwargdispatch(BaseDecorator, singledispatchmethod):
    """Extends singledispatch to allow kwargs to be used for dispatching.

    The normal single dispatching requires at least one arg for dispatching. This object retains this functionality, but
    allows the first kwarg to be used for dispatching if no args are provided. Furthermore, a kwarg name can be
    specified to have the dispatcher use that kwarg instead of the first kwarg.
    """
```

### Key Attributes

- `_kwarg`: The name of the kwarg to use for parsing the args for the class to use for dispatching.
- `_parse_method`: The default method for parsing the args for the class to use for dispatching.
- `_default_parse`: The default class to use for dispatching if the kwarg is not found.
- `parse`: The method for parsing the args for the class to use for dispatching.
- `arg_position`: The index of the arg to use for dispatching.
- `registry`: The registry of functions to dispatch to.
- `dispatch_cache`: The cache of functions to dispatch to.
- `cache_token`: The cache token for the dispatch function.
- `dispatch_function`: The function which dispatches to the correct function.

### Key Methods

#### Constructor

```python
def __init__(self, func=None, kwarg=None, *args, init=True, **kwargs):
    """Initialize a singlekwargdispatch decorator.
    
    Args:
        func: The function to wrap.
        kwarg: Either the name of kwarg to dispatch with or the method to wrap.
        *args: Arguments for inheritance.
        init: Determines if this object will construct.
        **kwargs: Keyword arguments for inheritance.
    """
```

#### Registration

```python
def register(self, cls, func=None):
    """Register a function for a type or union of types.
    
    Args:
        cls: The type or union of types to register for.
        func: The function to register.
        
    Returns:
        The registered function.
    """
```

#### Dispatching

```python
def dispatch(self, cls):
    """Return the function registered for the given class.
    
    Args:
        cls: The class to dispatch to.
        
    Returns:
        The function registered for the given class.
    """
```

```python
def dispatch_call(self, *args, **kwargs):
    """Parse input to decide which method to use in the registry.
    
    Args:
        *args: Positional arguments to pass to the found method.
        **kwargs: Keyword arguments to pass to the found method.
        
    Returns:
        The return of the found method.
    """
```

#### Parameter Parsing

```python
def parse_first(self, args, kwargs, is_method=False):
    """Parse input for the first arg or the first kwarg's class to be used for dispatching.
    
    Args:
        args: Positional arguments given to the method.
        kwargs: Keyword arguments given to the method.
        is_method: Determines if this is a method. If True, the first arg is not used.
        
    Returns:
        The class to be used for dispatching.
    """
```

```python
def parse_kwarg(self, args, kwargs, is_method=False):
    """Parse input for the first arg or a specific kwarg's class to be used for dispatching.
    
    Args:
        args: Positional arguments given to the method.
        kwargs: Keyword arguments given to the method.
        is_method: Determines if this is a method. If True, the first arg is not used.
        
    Returns:
        The class to be used for dispatching.
    """
```

#### Setters

```python
def set_kwarg(self, kwarg):
    """Set the name of the kwarg for dispatching and changes the arg parsing to check for the kwarg.
    
    Args:
        kwarg: The name of the kwarg or None for checking the first kwarg.
    """
```

### Usage Patterns

1. **Basic Usage**: Decorate a function with `@singlekwargdispatch` and register implementations for different types.

```python
@singlekwargdispatch
def func(x):
    return f"Default: {x}"

@func.register
def _(x: int):
    return f"Integer: {x}"

@func.register(str)
def _(x: str):
    return f"String: {x}"
```

2. **Keyword Argument Dispatching**: Specify a keyword argument to use for dispatching.

```python
@singlekwargdispatch(kwarg="format_as")
def format_data(data, format_as=None):
    return f"Default: {data}"

@format_data.register
def _(data, format_as: str = None):
    return f"String format: {data}"

@format_data.register
def _(data, format_as: int = None):
    return f"Integer format: {data}"
```

3. **Method Dispatching**: Use `singlekwargdispatch` with methods in classes.

```python
class MyClass:
    @singlekwargdispatch
    def process(self, data):
        return f"Default: {data}"
    
    @process.register
    def _(self, data: int):
        return f"Integer: {data}"
    
    @process.register
    def _(self, data: str):
        return f"String: {data}"
```

4. **Union Type Dispatching**: Register a single implementation for multiple types.

```python
@singlekwargdispatch
def process(data):
    return f"Default: {data}"

@process.register
def _(data: Union[int, float]):
    return f"Numeric: {data}"

@process.register
def _(data: Union[list, tuple]):
    return f"Sequence: {data}"
```

5. **Coroutine Dispatching**: Use `singlekwargdispatch` with async functions.

```python
@singlekwargdispatch
async def process_async(data):
    return f"Default: {data}"

@process_async.register
async def _(data: int):
    return f"Integer: {data}"

@process_async.register
async def _(data: str):
    return f"String: {data}"
```

For more details, refer to the source code and the examples provided in this tutorial.

## Troubleshooting / FAQs

### Q: How is singlekwargdispatch different from singledispatch?

A: The standard `singledispatch` from the `functools` module dispatches based on the type of the first positional argument. `singlekwargdispatch` extends this functionality to:

1. Dispatch based on the first positional argument (like `singledispatch`)
2. Dispatch based on the first keyword argument if no positional arguments are provided
3. Dispatch based on a specific keyword argument if specified with the `kwarg` parameter

This makes `singlekwargdispatch` more flexible, especially for functions that primarily use keyword arguments.

### Q: Why am I getting a TypeError when calling my function with no arguments?

A: `singlekwargdispatch` requires at least one argument (positional or keyword) to dispatch on. If you call a function decorated with `singlekwargdispatch` without any arguments, it will raise a `TypeError` because there's no argument to determine the type for dispatching.

```python
@singlekwargdispatch
def func():  # No parameters
    return "Default"

# This will raise TypeError: No args or kwargs given to dispatch.
func()
```

To avoid this, ensure your function has at least one parameter, and provide a value when calling it.

### Q: How do I specify which keyword argument to use for dispatching?

A: You can specify the keyword argument to use for dispatching by passing the `kwarg` parameter to the `singlekwargdispatch` decorator:

```python
@singlekwargdispatch(kwarg="format_as")
def format_data(data, format_as=None):
    return f"Default: {data}"
```

This tells `singlekwargdispatch` to use the `format_as` parameter for dispatching, rather than the first keyword argument.

### Q: Can I use singlekwargdispatch with methods in a class?

A: Yes, `singlekwargdispatch` works well with methods in classes. When using it with methods, remember that the first argument is always `self`, so the dispatching will be based on the second argument (or the specified keyword argument):

```python
class MyClass:
    @singlekwargdispatch
    def process(self, data):
        return f"Default: {data}"
    
    @process.register
    def _(self, data: int):
        return f"Integer: {data}"
```

### Q: How do I register multiple implementations for the same function?

A: You can register multiple implementations for the same function using the `register` method or decorator. Each implementation should handle a different type:

```python
@singlekwargdispatch
def process(data):
    return f"Default: {data}"

@process.register
def _(data: int):
    return f"Integer: {data}"

@process.register
def _(data: str):
    return f"String: {data}"

# You can also register directly with the type
@process.register(list)
def _(data):
    return f"List: {data}"
```

### Q: Can I register a single implementation for multiple types?

A: Yes, you can register a single implementation for multiple types using Union types or by registering the same function for multiple types:

```python
# Using Union type
@process.register
def _(data: Union[int, float]):
    return f"Numeric: {data}"

# Or registering the same function for multiple types
@process.register(tuple)
@process.register(list)
def _(data):
    return f"Sequence: {data}"
```

### Q: How do I handle None values in my dispatched functions?

A: You can register a specific implementation for `None` values using the `NoneType` from the `types` module:

```python
from types import NoneType

@singlekwargdispatch
def process(data):
    return f"Default: {data}"

@process.register
def _(data: NoneType):
    return "None value"
```

### Q: What happens if I register multiple implementations for the same type?

A: If you register multiple implementations for the same type, the last one registered will be used. Earlier registrations for the same type will be overwritten:

```python
@singlekwargdispatch
def process(data):
    return f"Default: {data}"

@process.register
def _(data: int):
    return f"First int implementation: {data}"

@process.register
def _(data: int):
    return f"Second int implementation: {data}"  # This one will be used
```

### Q: Can I use singlekwargdispatch with async functions?

A: Yes, `singlekwargdispatch` works seamlessly with async functions. Just make sure all your implementations are also async:

```python
@singlekwargdispatch
async def process_async(data):
    return f"Default: {data}"

@process_async.register
async def _(data: int):
    return f"Integer: {data}"
```

### Q: Is there a performance penalty for using singlekwargdispatch?

A: Yes, there is a small performance penalty compared to direct function calls or manual type checking, as `singlekwargdispatch` needs to determine the type and look up the appropriate implementation at runtime. However, the penalty is usually negligible for most applications, and the benefits of cleaner, more maintainable code often outweigh the performance cost.

If performance is critical, consider using direct function calls or manual type checking in performance-critical sections of your code.

### Q: How does singlekwargdispatch handle inheritance?

A: `singlekwargdispatch` respects the inheritance hierarchy when dispatching. If there's no exact match for a type, it will look for the closest matching type in the inheritance hierarchy:

```python
class Animal:
    pass

class Dog(Animal):
    pass

@singlekwargdispatch
def make_sound(animal):
    return "Unknown sound"

@make_sound.register
def _(animal: Animal):
    return "Generic animal sound"

# A Dog instance will use the Animal implementation if there's no specific Dog implementation
dog = Dog()
make_sound(dog)  # Returns "Generic animal sound"
```

This behavior is consistent with how `singledispatch` handles inheritance.

## Conclusion and Next Steps

In this tutorial, we've explored the `singlekwargdispatch` decorator, which extends Python's built-in `singledispatch` functionality to allow keyword arguments to be used for dispatching. We've covered:

1. **Basic Functionality**: How to use `singlekwargdispatch` for dispatching based on the type of the first positional argument, just like `singledispatch`.

2. **Keyword Argument Dispatching**: How to dispatch based on the first keyword argument when no positional arguments are provided.

3. **Specified Keyword Argument**: How to specify a particular keyword argument to use for dispatching.

4. **Method Dispatching**: How to use `singlekwargdispatch` with methods in classes.

5. **Advanced Features**: Working with Union types, None types, coroutines, and handling edge cases.

6. **Real-World Examples**: Practical examples of how `singlekwargdispatch` can be used in real-world scenarios.

7. **API Highlights**: A summary of the key components of the `singlekwargdispatch` API.

8. **Troubleshooting**: Common issues and questions when working with `singlekwargdispatch`.

The `singlekwargdispatch` decorator provides a powerful way to create functions that can handle different types of inputs with type-specific implementations. By extending the functionality of `singledispatch` to work with keyword arguments, it offers more flexibility and makes it easier to create clean, maintainable code.

### Key Takeaways

- `singlekwargdispatch` extends `singledispatch` to work with keyword arguments
- It can dispatch based on the first positional argument, the first keyword argument, or a specified keyword argument
- It works seamlessly with methods in classes and with coroutines
- It supports Union types, allowing a single implementation for multiple types
- It respects the inheritance hierarchy when dispatching

### Next Steps

Now that you understand how to use `singlekwargdispatch`, here are some suggestions for next steps:

1. **Explore the Source Code**: Dive into the source code of `singlekwargdispatch` to understand how it works under the hood.

2. **Try with Your Own Projects**: Identify places in your own code where `singlekwargdispatch` could simplify complex type-based logic.

3. **Combine with Other Decorators**: Experiment with combining `singlekwargdispatch` with other decorators like `lru_cache` or `wraps`.

4. **Explore Other Dispatching Mechanisms**: Compare `singlekwargdispatch` with other dispatching mechanisms like the visitor pattern or pattern matching.

5. **Contribute to the Project**: If you find bugs or have ideas for improvements, consider contributing to the baseobjects project.

Remember that while `singlekwargdispatch` is a powerful tool, it's not always the best solution for every problem. Consider the specific needs of your project and choose the approach that best fits your requirements.

Happy coding!